# Plots

In [ ]:
import os

import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import librosa
import seaborn as sns
from sklearn.model_selection import train_test_split
from IPython.display import display, Audio

In [ ]:
sns.set_theme(
    style="whitegrid",
)

## EARS Plots

In [ ]:
def create_dataframes(shared_path: str, kaggle: bool = False, fast: bool = True) -> dict[str, pd.DataFrame]:
    """Load and preprocess metadata for EARS and WHAM datasets."""
    if kaggle:
        COMMON_PATH = os.path.join(shared_path, "speech-enhancement")
        EARS_DATASET = os.path.join(COMMON_PATH, "ears_dataset", "ears_dataset", "speaker_statistics.json")
        EARS_FILES = os.path.join(COMMON_PATH, "ears_dataset", "ears_dataset", "ears_dataset_resampled")
        WHAM_DATA_TT = os.path.join(COMMON_PATH, "wham_noise", "wham_noise", "metadata", "mix_param_meta_tt.csv")
        WHAM_DATA_TR = os.path.join(COMMON_PATH, "wham_noise", "wham_noise", "metadata", "mix_param_meta_tr.csv")
        WHAM_DATA_CV = os.path.join(COMMON_PATH, "wham_noise", "wham_noise", "metadata", "mix_param_meta_cv.csv")
        WHAM_DATA_NOISE_TT = os.path.join(COMMON_PATH, "wham_noise", "wham_noise", "metadata", "noise_meta_tt.csv")
        WHAM_DATA_NOISE_TR = os.path.join(COMMON_PATH, "wham_noise", "wham_noise", "metadata", "noise_meta_tr.csv")
        WHAM_DATA_NOISE_CV = os.path.join(COMMON_PATH, "wham_noise", "wham_noise", "metadata", "noise_meta_cv.csv")
        WHAM_FILES_TT = os.path.join(COMMON_PATH, "wham_noise", "wham_noise", "resampled_tt")
        WHAM_FILES_CV = os.path.join(COMMON_PATH, "wham_noise", "wham_noise", "resampled_cv")
        WHAM_FILES_TR = os.path.join(COMMON_PATH, "wham_noise", "wham_noise", "resampled_tr")
    else:
        COMMON_PATH = shared_path
        EARS_DATASET = os.path.join(COMMON_PATH, "ears_dataset", "speaker_statistics.json")
        EARS_FILES = os.path.join(COMMON_PATH, "ears_dataset", "ears_dataset_resampled")
        WHAM_DATA_TT = os.path.join(COMMON_PATH, "wham_noise", "metadata", "mix_param_meta_tt.csv")
        WHAM_DATA_TR = os.path.join(COMMON_PATH, "wham_noise", "metadata", "mix_param_meta_tr.csv")
        WHAM_DATA_CV = os.path.join(COMMON_PATH, "wham_noise", "metadata", "mix_param_meta_cv.csv")
        WHAM_DATA_NOISE_TT = os.path.join(COMMON_PATH, "wham_noise", "metadata", "noise_meta_tt.csv")
        WHAM_DATA_NOISE_TR = os.path.join(COMMON_PATH, "wham_noise", "metadata", "noise_meta_tr.csv")
        WHAM_DATA_NOISE_CV = os.path.join(COMMON_PATH, "wham_noise", "metadata", "noise_meta_cv.csv")
        WHAM_FILES_TT = os.path.join(COMMON_PATH, "wham_noise", "resampled_tt")
        WHAM_FILES_CV = os.path.join(COMMON_PATH, "wham_noise", "resampled_cv")
        WHAM_FILES_TR = os.path.join(COMMON_PATH, "wham_noise", "resampled_tr")

    personal_metadata_df = get_ears_personal_metadata(EARS_DATASET)
    ears_metadata_df = preprocess_ears_metadata(EARS_FILES, fast=fast, verbose=False)
    wham_df = preprocess_wham_metadata(
        wham_data_cv=WHAM_DATA_CV,
        wham_data_tr=WHAM_DATA_TR,
        wham_data_tt=WHAM_DATA_TT,
        wham_files_cv=WHAM_FILES_CV,
        wham_files_tr=WHAM_FILES_TR,
        wham_files_tt=WHAM_FILES_TT,
        wham_noise_cv=WHAM_DATA_NOISE_CV,
        wham_noise_tr=WHAM_DATA_NOISE_TR,
        wham_noise_tt=WHAM_DATA_NOISE_TT,
        fast=fast,
        verbose=False
    )
    ears_df = merge_ears_filepaths_with_metadata(ears_metadata_df, personal_metadata_df)
    return {"wham_df": wham_df, "ears_df": ears_df}

def merge_ears_filepaths_with_metadata(paths_df: pd.DataFrame, meta_df: pd.DataFrame, verbose: bool = False) -> pd.DataFrame:
    """
    Merge EARS file paths DataFrame with EARS person metadata DataFrame on 'person' column.

    Args:
        paths_df (pd.DataFrame): DataFrame containing file paths with 'person' column.
        meta_df (pd.DataFrame): DataFrame containing EARS person metadata with index as 'person'.

    Returns:
        pd.DataFrame: Merged DataFrame containing file paths and corresponding person metadata.
    """
    merged_df = pd.merge(paths_df, meta_df, left_on="person", right_index=True, how="inner")
    if verbose:
        print("Merged EARS Metadata Summary:")
        print(merged_df.head())
    return merged_df


def get_ears_personal_metadata(path: str) -> pd.DataFrame:
    """
    Load EARS person metadata from a JSON file.
    
    Args:
        path (str): Path to the JSON file containing EARS person metadata.

    Returns:
        pd.DataFrame: DataFrame containing the EARS person metadata.
    """
    meta_df = pd.read_json(path).T
    return meta_df


def preprocess_ears_metadata(path: str, fast: bool = True, verbose: bool = False) -> pd.DataFrame:
    """
    Preprocess EARS dataset metadata. Adds columns for length, style, emotion, freeform speech, and non-speech.

    Args:
        path (str): Path to the EARS dataset.
        fast (bool, optional): If True, skip length calculation for speed. Defaults to True.
        verbose (bool, optional): If True, print summary statistics. Defaults to False.

    Returns:
        pd.DataFrame: Preprocessed EARS metadata DataFrame.
    """
    df = _create_ears_paths(path)

    if not fast:
        df["length [s]"] = df["path"].apply(_count_len)

    df["style"] = df["file"].apply(_categorize_style)
    df["emotion"] = df["file"].apply(_categorize_emotion)
    df["is_freeform_speech"] = df["file"].apply(_categorize_freeform_speech)
    df["is_non_speech"] = df["file"].apply(_categorize_non_speech)

    df["style"] = df["style"].astype("category")
    df["emotion"] = df["emotion"].astype("category")

    if verbose:
        print("EARS Metadata Preprocessing Summary:")
        print(df.head())
    return df


def preprocess_wham_metadata(
        *,
        wham_data_cv: str,
        wham_data_tt: str,
        wham_data_tr: str,
        wham_noise_cv: str,
        wham_noise_tt: str,
        wham_noise_tr: str,
        wham_files_cv: str,
        wham_files_tt: str,
        wham_files_tr: str,
        fast: bool = True,
        verbose: bool = False
) -> pd.DataFrame:
    """
    Preprocess WHAM dataset metadata by merging data and noise CSV files and adding file paths.

    Args:
        wham_data_cv (str): Path to WHAM metadata cv CSV file.
        wham_data_tt (str): Path to WHAM metadata tt CSV file.
        wham_data_tr (str): Path to WHAM metadata tr CSV file.
        wham_noise_cv (str): Path to WHAM noise cv CSV file.
        wham_noise_tt (str): Path to WHAM noise tt CSV file.
        wham_noise_tr (str): Path to WHAM noise tr CSV file.
        wham_files_cv (str): Path to WHAM cv audio files directory.
        wham_files_tt (str): Path to WHAM tt audio files directory.
        wham_files_tr (str): Path to WHAM tr audio files directory.
        fast (bool, optional): If True, skip length calculation for speed. Defaults to True.
        verbose (bool, optional): If True, print summary statistics. Defaults to False.

    Returns:
        pd.DataFrame: Preprocessed WHAM metadata DataFrame.
    """
    wham_data_cv_df = pd.read_csv(wham_data_cv)
    wham_data_tt_df = pd.read_csv(wham_data_tt)
    wham_data_tr_df = pd.read_csv(wham_data_tr)

    wham_noise_cv_df = pd.read_csv(wham_noise_cv)
    wham_noise_tt_df = pd.read_csv(wham_noise_tt)
    wham_noise_tr_df = pd.read_csv(wham_noise_tr)

    wham_data_cv_df["path"] = wham_data_cv_df["utterance_id"].map(lambda f: os.path.join(wham_files_cv, f))
    wham_data_tt_df["path"] = wham_data_tt_df["utterance_id"].map(lambda f: os.path.join(wham_files_tt, f))
    wham_data_tr_df["path"] = wham_data_tr_df["utterance_id"].map(lambda f: os.path.join(wham_files_tr, f))

    wham_noise_df = pd.concat([wham_noise_tr_df, wham_noise_tt_df, wham_noise_cv_df], ignore_index=True)
    wham_data_df = pd.concat([wham_data_tr_df, wham_data_tt_df, wham_data_cv_df], ignore_index=True)
    wham_df = pd.merge(wham_noise_df, wham_data_df, on="utterance_id", how="inner")
    if not fast:
        wham_df["len"] = wham_df["path"].map(_count_len)

    if verbose:
        print("WHAM Metadata Preprocessing Summary:")
        print(wham_df.head())
    return wham_df


def prepare_for_training(df: pd.DataFrame, train_percentage: float, reduce_to: float | int | None, filter_to: dict[str, list[str]] | None = None, verbose: bool = False) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Prepare dataset by splitting into train, validate, and test sets. Also reduces the size of each set based on the reduce_to parameter.

    Args:
        df (pd.DataFrame): The input dataframe to split.
        train_percentage (float): The percentage of data to use for training set. Defaults to 0.8 (80%). Test and validation sets will each get half of the remaining data.
        reduce_to (float | int | None): If float in (0,1], reduces the dataset to that fraction. If int > 1, reduces the dataset to that many samples. If None, no reduction is applied.
        filter_to (dict[str, list[str]] | None): A dictionary where keys are column names and values are lists of acceptable values for those columns. If provided, filters the dataset before splitting. Defaults to None.
        verbose (bool): If True, prints the sizes of the resulting datasets.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]: A tuple containing the train, validate, and test dataframes.
    """
    if filter_to is not None:
        df = _filter_dataset(df, filter_to)

    train_df, temp_df = train_test_split(df, train_size=train_percentage, random_state=42)
    validate_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

    train_df = _reduce_dataset(train_df, reduce_to)
    validate_df = _reduce_dataset(validate_df, reduce_to)
    test_df = _reduce_dataset(test_df, reduce_to)
    if verbose:
        print(f"Train set size: {len(train_df)}")
        print(f"Validate set size: {len(validate_df)}")
        print(f"Test set size: {len(test_df)}")
    return train_df, validate_df, test_df


# ==============================================================================
#                       CATEGORY: PREPROCESS METADATA
#                 Helper functions for preprocessing metadata
# ==============================================================================


# ------------------------------------------------------------------------------
# EARS Dataset - Helper Functions
# ------------------------------------------------------------------------------
    
def _create_ears_paths(ears_dataset_path: str) -> pd.DataFrame:
    """
    Create a DataFrame with path, file, person information to all .wav files in the EARS dataset.

    Args:
        ears_dataset_path (str): Path to the EARS dataset.

    Returns:
        pd.DataFrame: DataFrame containing 'person', 'file', and 'path' columns for each .wav file.
    """
    rows = []
    max_person_index = 107
    for i in range(max_person_index):
        person = "p" + f"{i+1}".zfill(3) + "_resampled"
        person_dir = os.path.join(ears_dataset_path, person)
        
        if not os.path.exists(person_dir):
            raise ValueError(f"Path does not exists: {person_dir}")
        
        for file in os.listdir(person_dir):
            if file.endswith(".wav"):
                rows.append(
                    {
                        "person": person.replace("_resampled", ""),
                        "file": file,
                        "path": os.path.join(person_dir, file)
                    }
                )
        
    return pd.DataFrame(rows)

def _categorize_style(filename: str) -> str:
    """
    Categorize the speaking style based on given information in the filename.

    Args:
        filename (str): The filename to categorize.

    Returns:
        str: The categorized style from given set or "other" if none match.
    """
    styles = {"regular", "loud", "whisper", "fast", "slow", "highpitch", "lowpitch"}
    return next((s for s in styles if s in filename), "normal")

def _categorize_emotion(filename: str) -> str:
    """
    Categorize the emotion based on given information in the filename.

    Args:
        filename (str): The filename to categorize.

    Returns:
        str: The categorized emotion from given set or "other" if none match.
    """
    emotions = {'adoration', 'fear', 'pain', 'realization', 'confusion',
       'cuteness', 'distress', 'guilt', 'amazement', 'contentment',
       'pride', 'desire', 'relief', 'disappointment', 'disgust',
       'embarassment', 'anger', 'interest', 'serenity', 'sadness',
       'amusement', 'extasy', 'neutral'}
    return next((e for e in emotions if e in filename), "normal")
    
def _categorize_freeform_speech(filename: str) -> bool:
    """
    Categorize if the filename corresponds to clean long freeform speech.

    Args:
        filename (str): The filename to categorize.

    Returns:
        bool: True if the filename corresponds to freeform speech, False otherwise.
    """
    return True if "freeform_speech" in filename else False

def _categorize_non_speech(filename: str) -> bool:
    """
    Categorize if the filename corresponds to non-speech sounds.
    
    Args:
        filename (str): The filename to categorize.
        
    Returns:
        bool: True if the filename corresponds to non-speech sounds, False otherwise.
    """
    non_speech = {'interjection_greetings', 'vegetative_throat',
       'nonverbal_screaming', 'nonverbal_crying', 'vegetative_eating',
       'melodic_happy_birthday', 'vegetative_yawning',
       'nonverbal_laughter_open', 'nonverbal_cheering',
       'nonverbal_yelling', 'vegetative_coughing',
       'nonverbal_laughter_closed', 'interjection_agreement',
       'interjection_filler', 'vegetative_sneezing',
       'interjection_congratulations'}
    return any(ns in filename for ns in non_speech)


# ------------------------------------------------------------------------------
# WHAM Dataset - Helper Functions
# ------------------------------------------------------------------------------


# ------------------------------------------------------------------------------
# Common - Helper Functions
# ------------------------------------------------------------------------------

def _count_len(file: str) -> float:
    """
    Count the length of a .wav file in seconds.
    
    Args:
        file (str): Path to the .wav file.

    Returns:
        float: Length of the audio file in seconds.
    """
    data, samplerate = sf.read(file)
    return len(data) / samplerate


def _reduce_dataset(df: pd.DataFrame, reduce_param: float | int | None) -> pd.DataFrame:
    """
    Reduces the dataset based on the reduce_param.
    
    Args:
        df (pd.DataFrame): The input dataframe to reduce.
        reduce_param (float | int | None): If float in (0,1], reduces the dataset to that fraction. If int > 1, reduces the dataset to that many samples. If None, no reduction is applied.

    Returns:
        pd.DataFrame: The reduced dataframe.
    """
    if reduce_param is None:
        return df
    if isinstance(reduce_param, float) and 0 < reduce_param <= 1:
        return df.sample(frac=reduce_param, random_state=42)
    if isinstance(reduce_param, int) and reduce_param > 1:
        return df.sample(n=min(reduce_param, len(df)), random_state=42)
    raise ValueError("reduce_param musi być None, float w (0,1] albo int > 1")


def _filter_dataset(df: pd.DataFrame, filter: dict[str, list[str]]) -> pd.DataFrame:
    """
    Filters the dataset based on the provided filter criteria.

    Args:
        df (pd.DataFrame): The input dataframe to filter.
        filter (dict[str, list[str]]): A dictionary where keys are column names and values are lists of acceptable values for those columns.

    Returns:
        pd.DataFrame: The filtered dataframe.
    """
    for column, accepted_values in filter.items():
        df = df[df[column].isin(accepted_values)]
    return df

In [ ]:
dfs = create_dataframes("../..", fast=False)

In [ ]:
ears_df = dfs["ears_df"]
wham_df = dfs["wham_df"]

## EARS plots

In [ ]:
ears_df

In [ ]:
print("EARS Dataset Summary:")
print(f"Total samples: {len(ears_df)}")
print(f"Unique speakers: {ears_df['person'].nunique()}")
print(f"Speaking styles: {len(ears_df['style'].value_counts())}")
print(f"Emotions: {len(ears_df['emotion'].value_counts())}")
print(f"Freeform speech samples: {ears_df['is_freeform_speech'].sum()}")
print(f"Non-speech samples: {ears_df['is_non_speech'].sum()}")
print(f"Total audio length: {ears_df['length [s]'].sum() / 3600:.2f} hours")
print(f"Total audio length: {ears_df['length [s]'].sum():.2f} seconds")
ears_freeform = ears_df[ears_df["is_freeform_speech"]]
print(f"Total audio length freeform speech: {ears_freeform['length [s]'].sum() / 3600:.2f} hours")
print(f"Total audio length freeform speech: {ears_freeform['length [s]'].sum():.2f} seconds")
ears_non_speech = ears_df[ears_df["is_non_speech"]]
print(f"Total audio length non-speech: {ears_non_speech['length [s]'].sum() / 3600:.2f} hours")
print(f"Total audio length non-speech: {ears_non_speech['length [s]'].sum():.2f} seconds")

In [ ]:
def plot_categorical_column(series: pd.Series, ax: plt.Axes) -> None:
    """Bar chart for a categorical column (sample count per value)."""
    counts = series.value_counts().sort_values(ascending=False)
    n_unique = series.nunique()
    sns.barplot(x=counts.index.astype(str), y=counts.values, ax=ax)
    ax.set_title(f"{series.name}\n(categorical – bar chart, {n_unique} values)")
    ax.set_xlabel("Value")
    ax.set_ylabel("Sample count")
    ax.tick_params(axis="x", rotation=45)
    for bar in ax.patches:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(counts.values) * 0.01,
            f"{int(bar.get_height())}",
            ha="center", va="bottom", fontsize=8,
        )


def plot_numeric_column(series: pd.Series, ax: plt.Axes) -> None:
    """Histogram for a numeric column with mean line."""
    sns.histplot(series.dropna(), ax=ax)
    ax.set_title(f"{series.name}\n(numeric – histogram)")
    ax.set_xlabel("Value")
    ax.set_ylabel("Sample count")
    mean_val = series.mean()
    ax.axvline(mean_val, color="red", linestyle="--", linewidth=1.2, label=f"mean: {mean_val:.2f}")
    ax.legend(fontsize=8)


def plot_high_cardinality_column(series: pd.Series, ax: plt.Axes, top_n: int = 20) -> None:
    """Horizontal bar chart for top-N values of a high-cardinality column."""
    n_unique = series.nunique()
    counts = series.value_counts().head(top_n)
    sns.barplot(x=counts.values, y=counts.index.astype(str), ax=ax, orient="h")
    ax.set_title(f"{series.name}\n(high cardinality – top {top_n} of {n_unique})")
    ax.set_xlabel("Sample count")
    ax.set_ylabel("Value")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
plot_high_cardinality_column(ears_df["person"], ax, top_n=25)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_numeric_column(ears_df["length [s]"], ax)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_categorical_column(ears_df["emotion"], ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_categorical_column(ears_df["style"], ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_categorical_column(ears_df["age"], ax)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_categorical_column(ears_df["ethnicity"], ax)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_categorical_column(ears_df["gender"], ax)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_categorical_column(ears_df["weight"], ax)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_categorical_column(ears_df["native language"], ax)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_categorical_column(ears_df["height"], ax)
plt.tight_layout()
plt.show()


## WHAM

In [ ]:
wham_df

In [ ]:
print("WHAM Dataset Summary:")
print(f"Total samples: {len(wham_df)}")
print(f"Unique files: {wham_df['File ID'].nunique()}")
print(f"Unique locations: {len(wham_df['Location ID'].value_counts())}")
print(f"Unique location days: {len(wham_df['Location Day ID'].value_counts())}")
print(f"Total audio length: {wham_df['len'].sum() / 3600:.2f} hours")
print(f"Total audio length: {wham_df['len'].sum():.2f} seconds")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_numeric_column(wham_df["len"], ax)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_categorical_column(wham_df["Location ID"], ax)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_categorical_column(wham_df["Location Day ID"], ax)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_categorical_column(wham_df["File ID"], ax)
plt.tight_layout()
plt.show()

## EARS Audio

In [ ]:
def play_person_audio(
    df: pd.DataFrame,
    person: str,
    style: str | None = None,
    emotion: str | None = None,
    max_files: int = 10,
    shuffle: bool = True
) -> None:
    """Display IPython audio players for a selected person from ears_df, optionally filtered by style and emotion."""
    subset = df[df["person"] == person]

    if shuffle:
        subset = subset.sample(frac=1).reset_index(drop=True)

    if style is not None:
        subset = subset[subset["style"] == style]
    if emotion is not None:
        subset = subset[subset["emotion"] == emotion]

    subset = subset.head(max_files)

    for _, row in subset.iterrows():
        display(f"Style: {row['style']}, Emotion: {row['emotion']}, Is freeform: {row['is_freeform_speech']}, Is non-speech: {row['is_non_speech']}, Length: {row['length [s]']:.2f} s")
        display(Audio(filename=row["path"], autoplay=False))

In [ ]:
play_person_audio(ears_df, person="p001", emotion=None, max_files=5)

## WHAM Audio

In [ ]:
def play_noise_audio(
    df: pd.DataFrame,
    file_id: str | None = None,
    location_id: str | None = None,
    day_id: str | None = None,
    max_files: int = 10,
    shuffle: bool = True
) -> None:
    """Display IPython audio players for wham_df, optionally filtered by file_id, location_id, and day_id."""
    subset = df

    if shuffle:
        subset = subset.sample(frac=1).reset_index(drop=True)

    if file_id is not None:
        subset = subset[subset["File ID"] == file_id]
    if location_id is not None:
        subset = subset[subset["Location ID"] == location_id]
    if day_id is not None:
        subset = subset[subset["Location Day ID"] == day_id]

    subset = subset.head(max_files)

    for _, row in subset.iterrows():
        display(f"File ID: {row['File ID']}, Location ID: {row['Location ID']}, Location Day ID: {row['Location Day ID']}, Length: {row['len']:.2f} s")
        display(Audio(filename=row["path"], autoplay=False))

In [ ]:
play_noise_audio(wham_df, max_files=10)

## Audio plots - utils

In [ ]:
import numpy as np

def plot_waveform(file: str, ax: plt.Axes) -> None:
    """Plot the waveform of a .wav file on the given Axes."""
    data, samplerate = sf.read(file)
    time = np.arange(len(data)) / samplerate
    ax.plot(time, data)
    ax.set_title(f"Waveform: {os.path.basename(file)}")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Amplitude")
    ax.set_xlim(0, time[-1])
    ax.grid(True)

def plot_spectrogram(file: str, ax: plt.Axes) -> None:
    """Plot the spectrogram of a .wav file using librosa on the given Axes."""
    data, samplerate = sf.read(file)
    S = librosa.stft(data.astype(float))
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)
    img = librosa.display.specshow(S_db, sr=samplerate, x_axis='time', y_axis='hz', ax=ax)
    ax.set_title(f"Spectrogram: {os.path.basename(file)}")
    fig.colorbar(img, ax=ax, format="%+2.0f dB")

def plot_mel_spectrogram(file: str, ax: plt.Axes) -> None:
    """Plot the Mel spectrogram of a .wav file using librosa on the given Axes."""
    data, samplerate = sf.read(file)
    S = librosa.feature.melspectrogram(y=data.astype(float), sr=samplerate, n_mels=128)
    S_db = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(S_db, sr=samplerate, x_axis='time', y_axis='mel', ax=ax)
    ax.set_title(f"Mel Spectrogram: {os.path.basename(file)}")
    fig.colorbar(img, ax=ax, format="%+2.0f dB")

## EARS Audio plots

In [ ]:
sample_files = ears_df["path"].sample(6, random_state=42).values

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 8))
for file, ax in zip(sample_files, axes.flatten()):
    plot_waveform(file, ax)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 8))
for file, ax in zip(sample_files, axes.flatten()):
    plot_spectrogram(file, ax)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 8))
for file, ax in zip(sample_files, axes.flatten()):
    plot_mel_spectrogram(file, ax)
plt.tight_layout()
plt.show()

# WHAM Audio plots

In [ ]:
sample_files = wham_df["path"].sample(n=8, random_state=42).tolist()

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(12, 8))
for file, ax in zip(sample_files, axes.flatten()):
    plot_waveform(file, ax)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(12, 8))
for file, ax in zip(sample_files, axes.flatten()):
    plot_spectrogram(file, ax)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(12, 8))
for file, ax in zip(sample_files, axes.flatten()):
    plot_mel_spectrogram(file, ax)
plt.tight_layout()
plt.show()

# Learning curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_learning_curves(csv_path: str):
    """Plot learning curves from a CSV file containing training metrics."""
    df = pd.read_csv(csv_path)

    metrics = ['val_loss', 'g_total', 'g_gan', 'g_mag', 'g_sc', 'd_loss']
    available_metrics = [m for m in metrics if m in df.columns]
    df_epoch = df.groupby('epoch')[available_metrics].mean().reset_index()

    if all(m in df_epoch for m in ['g_gan', 'g_mag', 'g_sc']):
        df_epoch['g_sum'] = df_epoch['g_gan'] + df_epoch['g_mag'] + df_epoch['g_sc']

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    fig.suptitle('Training plots MelReGAN', fontsize=16)

    if all(m in df_epoch for m in ['val_loss', 'g_gan', 'g_mag', 'g_sc']):
        axes[0].plot(df_epoch['epoch'], df_epoch['val_loss'], label='val_loss')
        axes[0].plot(df_epoch['epoch'], df_epoch['g_gan'], label='g_gan')
        axes[0].plot(df_epoch['epoch'], df_epoch['g_mag'], label='g_mag')
        axes[0].plot(df_epoch['epoch'], df_epoch['g_sc'], label='g_sc')
        
        axes[0].set_title('Learning plots')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].legend()

    if 'g_total' in df_epoch:
        axes[1].plot(df_epoch['epoch'], df_epoch['g_total'], label='G Total Loss')
        axes[1].set_title('Generator Total Loss')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Loss')
        axes[1].legend()

    if 'd_loss' in df_epoch and 'g_gan' in df_epoch:
        axes[2].plot(df_epoch['epoch'], df_epoch['d_loss'], label='d_loss (Discriminator)')
        axes[2].plot(df_epoch['epoch'], df_epoch['g_gan'], label='g_gan (Adversarial Loss)')
        axes[2].set_title('Discriminator vs Adversarial Loss')
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('Loss')
        axes[2].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
plot_learning_curves("train-metrics.csv")